<a href="https://colab.research.google.com/github/javisagredo-dev/Evaluacion2_Prog_DS/blob/main/02_aprendizaje_supervisado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aprendizaje supervisado: predicción de variedades de frijol

## Importaciones necesarias

In [30]:
import pickle
import pandas as pd
import numpy as np
from google.colab import drive
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import time
import warnings
warnings.filterwarnings('ignore')

## Carga y lectura de datos

In [31]:
drive.mount('/content/drive')
df = pd.read_excel('/content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/Evaluacion_2/Raw/Dry_Bean_Dataset.xlsx')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Copia del dataset para aprendizaje supervisado

In [32]:
df_model = df.copy()

## Separación de variables y codificación de etiquetas

In [33]:
X = df_model.drop('Class', axis=1)

le = LabelEncoder()
y = le.fit_transform(df_model['Class'])

print(f"Clases originales: {le.classes_}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Clases originales: ['BARBUNYA' 'BOMBAY' 'CALI' 'DERMASON' 'HOROZ' 'SEKER' 'SIRA']
X shape: (13611, 16)
y shape: (13611,)


## División en entrenamiento y prueba con estratificación

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(df_model)*100:.0f}%)")
print(f"Prueba: {X_test.shape[0]} muestras ({X_test.shape[0]/len(df_model)*100:.0f}%)")
print(f"\nBalance en entrenamiento:")
for i, clase in enumerate(le.classes_):
    n = sum(y_train == i)
    print(f"   {clase}: {n} ({n/len(y_train)*100:.1f}%)")

Entrenamiento: 10888 muestras (80%)
Prueba: 2723 muestras (20%)

Balance en entrenamiento:
   BARBUNYA: 1057 (9.7%)
   BOMBAY: 418 (3.8%)
   CALI: 1304 (12.0%)
   DERMASON: 2837 (26.1%)
   HOROZ: 1542 (14.2%)
   SEKER: 1621 (14.9%)
   SIRA: 2109 (19.4%)


## Estandarización de características con StandardScaler

In [35]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Escalado completado")
print(f"Media aproximada de X_train_scaled: {X_train_scaled.mean():.4f}")
print(f"Std aproximada de X_train_scaled: {X_train_scaled.std():.4f}")

Escalado completado
Media aproximada de X_train_scaled: 0.0000
Std aproximada de X_train_scaled: 1.0000


## Árbol de Decisión: entrenamiento y evaluación

In [36]:
start_time = time.time()

dt = DecisionTreeClassifier(random_state=42, max_depth=10)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt, average='weighted')
recall_dt = recall_score(y_test, y_pred_dt, average='weighted')
f1_dt = f1_score(y_test, y_pred_dt, average='weighted')
time_dt = time.time() - start_time

### Resultados

In [37]:
resultados = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Tiempo'],
    'Valor': [f"{accuracy_dt:.4f}", f"{precision_dt:.4f}", f"{recall_dt:.4f}", f"{f1_dt:.4f}", f"{time_dt:.2f}s"]
})
display(resultados)

,Métrica,Valor
0,Accuracy,0.9071
1,Precision,0.9074
2,Recall,0.9071
3,F1-Score,0.9072
4,Tiempo,0.50s


### Matriz de confusión

In [38]:
cm = confusion_matrix(y_test, y_pred_dt)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
display(cm_df)

,BARBUNYA,BOMBAY,CALI,DERMASON,HOROZ,SEKER,SIRA
BARBUNYA,238,0,15,0,0,3,9
BOMBAY,0,104,0,0,0,0,0
CALI,15,0,302,0,6,1,2
DERMASON,0,0,0,647,0,14,48
HOROZ,3,0,6,6,357,0,14
SEKER,3,0,1,8,0,381,13
SIRA,4,0,2,60,8,12,441


### Reporte por clase

In [39]:
print(classification_report(y_test, y_pred_dt, target_names=le.classes_))

              precision    recall  f1-score   support

    BARBUNYA       0.90      0.90      0.90       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.93      0.93      0.93       326
    DERMASON       0.90      0.91      0.90       709
       HOROZ       0.96      0.92      0.94       386
       SEKER       0.93      0.94      0.93       406
        SIRA       0.84      0.84      0.84       527

    accuracy                           0.91      2723
   macro avg       0.92      0.92      0.92      2723
weighted avg       0.91      0.91      0.91      2723



## Regresión Logística: entrenamiento y evaluación

In [40]:
start_time = time.time()

lr = LogisticRegression(random_state=42, max_iter=1000, solver='lbfgs')
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr, average='weighted')
recall_lr = recall_score(y_test, y_pred_lr, average='weighted')
f1_lr = f1_score(y_test, y_pred_lr, average='weighted')
time_lr = time.time() - start_time

### Resultados

In [41]:
resultados = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Tiempo'],
    'Valor': [f"{accuracy_lr:.4f}", f"{precision_lr:.4f}", f"{recall_lr:.4f}", f"{f1_lr:.4f}", f"{time_lr:.2f}s"]
})
display(resultados)

,Métrica,Valor
0,Accuracy,0.9214
1,Precision,0.9222
2,Recall,0.9214
3,F1-Score,0.9216
4,Tiempo,1.75s


### Matriz de Confusión

In [42]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_df_lr = pd.DataFrame(cm_lr, index=le.classes_, columns=le.classes_)
display(cm_df_lr)

,BARBUNYA,BOMBAY,CALI,DERMASON,HOROZ,SEKER,SIRA
BARBUNYA,236,0,18,0,0,4,7
BOMBAY,0,104,0,0,0,0,0
CALI,8,0,307,0,5,2,4
DERMASON,0,0,0,644,0,13,52
HOROZ,2,0,4,5,367,0,8
SEKER,2,0,0,5,0,387,12
SIRA,0,0,1,43,9,10,464


### Reporte por clase

In [43]:
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

              precision    recall  f1-score   support

    BARBUNYA       0.95      0.89      0.92       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.93      0.94      0.94       326
    DERMASON       0.92      0.91      0.92       709
       HOROZ       0.96      0.95      0.96       386
       SEKER       0.93      0.95      0.94       406
        SIRA       0.85      0.88      0.86       527

    accuracy                           0.92      2723
   macro avg       0.94      0.93      0.93      2723
weighted avg       0.92      0.92      0.92      2723



## Máquina de Soporte Vectorial (SVM): entrenamiento y evaluación

In [44]:
start_time = time.time()

svm = SVC(kernel='linear', random_state=42, C=1.0)
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm, average='weighted')
recall_svm = recall_score(y_test, y_pred_svm, average='weighted')
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')
time_svm = time.time() - start_time

### Resultados

In [45]:
resultados = pd.DataFrame({
    'Métrica': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Tiempo'],
    'Valor': [f"{accuracy_svm:.4f}", f"{precision_svm:.4f}", f"{recall_svm:.4f}", f"{f1_svm:.4f}", f"{time_svm:.2f}s"]
})
display(resultados)

,Métrica,Valor
0,Accuracy,0.9221
1,Precision,0.9222
2,Recall,0.9221
3,F1-Score,0.9221
4,Tiempo,1.83s


### Matriz de Confusión

In [46]:
cm_svm = confusion_matrix(y_test, y_pred_svm)
cm_df_svm = pd.DataFrame(cm_svm, index=le.classes_, columns=le.classes_)
display(cm_df_svm)

,BARBUNYA,BOMBAY,CALI,DERMASON,HOROZ,SEKER,SIRA
BARBUNYA,238,0,17,0,1,4,5
BOMBAY,1,103,0,0,0,0,0
CALI,9,0,308,0,5,2,2
DERMASON,0,0,0,651,0,11,47
HOROZ,2,0,6,6,367,0,5
SEKER,1,0,0,5,0,389,11
SIRA,2,0,1,52,9,8,455


### Reporte por clase

In [47]:
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))

              precision    recall  f1-score   support

    BARBUNYA       0.94      0.90      0.92       265
      BOMBAY       1.00      0.99      1.00       104
        CALI       0.93      0.94      0.94       326
    DERMASON       0.91      0.92      0.91       709
       HOROZ       0.96      0.95      0.96       386
       SEKER       0.94      0.96      0.95       406
        SIRA       0.87      0.86      0.87       527

    accuracy                           0.92      2723
   macro avg       0.94      0.93      0.93      2723
weighted avg       0.92      0.92      0.92      2723



## Comparación de modelos

In [48]:
comparison = pd.DataFrame({
    'Modelo': ['Árbol de Decisión', 'Regresión Logística', 'SVM (Lineal)'],
    'Accuracy': [accuracy_dt, accuracy_lr, accuracy_svm],
    'Precision': [precision_dt, precision_lr, precision_svm],
    'Recall': [recall_dt, recall_lr, recall_svm],
    'F1-Score': [f1_dt, f1_lr, f1_svm],
    'Tiempo (s)': [time_dt, time_lr, time_svm]
})

display(comparison.round(4))

best_model = comparison.loc[comparison['Accuracy'].idxmax(), 'Modelo']
best_accuracy = comparison['Accuracy'].max()
best_f1 = comparison.loc[comparison['Accuracy'].idxmax(), 'F1-Score']

print(f"\n{'='*60}")
print(f"MEJOR MODELO: {best_model}")
print(f"Accuracy: {best_accuracy:.4f}")
print(f"F1-Score: {best_f1:.4f}")
print(f"{'='*60}")

,Modelo,Accuracy,Precision,Recall,F1-Score,Tiempo (s)
0,Árbol de Decisión,0.9071,0.9074,0.9071,0.9072,0.4959
1,Regresión Logística,0.9214,0.9222,0.9214,0.9216,1.7475
2,SVM (Lineal),0.9221,0.9222,0.9221,0.9221,1.8339



MEJOR MODELO: SVM (Lineal)
Accuracy: 0.9221
F1-Score: 0.9221


## Verificación de sobreajuste (overfitting)

In [49]:
y_train_pred_dt = dt.predict(X_train)
y_train_pred_lr = lr.predict(X_train_scaled)
y_train_pred_svm = svm.predict(X_train_scaled)

train_acc_dt = accuracy_score(y_train, y_train_pred_dt)
train_acc_lr = accuracy_score(y_train, y_train_pred_lr)
train_acc_svm = accuracy_score(y_train, y_train_pred_svm)

overfit_df = pd.DataFrame({
    'Modelo': [' Árbol de Decisión', ' Regresión Logística', ' SVM (Lineal)'],
    'Train Acc': [train_acc_dt, train_acc_lr, train_acc_svm],
    'Test Acc': [accuracy_dt, accuracy_lr, accuracy_svm],
    'Diferencia': [train_acc_dt - accuracy_dt,
                   train_acc_lr - accuracy_lr,
                   train_acc_svm - accuracy_svm]
})

# Mostrar tabla con display
display(overfit_df.round(4))

# Agregar columna de diagnóstico con emojis
display(Markdown("###  Interpretación"))

for i, row in overfit_df.iterrows():
    diff = row['Diferencia']
    if diff < 0.02:
        estado = "Excelente (NO overfitting)"
    elif diff < 0.05:
        estado = "Aceptable"
    else:
        estado = "Overfitting"
    print(f"**{row['Modelo']}:** Diferencia = {diff:.4f} → {estado}")

# Leyenda
display(Markdown("---"))
display(Markdown("**Referencia:**"))
display(Markdown("- Diferencia < 0.02 → Excelente (NO overfitting)"))
display(Markdown("- Diferencia 0.02-0.05 → Aceptable"))
display(Markdown("- Diferencia > 0.05 → Overfitting"))

best_overfit = overfit_df.loc[overfit_df['Diferencia'].idxmin(), 'Modelo']
print(f"\n Modelo con menor sobreajuste: {best_overfit}")

,Modelo,Train Acc,Test Acc,Diferencia
0,Árbol de Decisión,0.9634,0.9071,0.0563
1,Regresión Logística,0.9262,0.9214,0.0048
2,SVM (Lineal),0.9292,0.9221,0.0070


###  Interpretación

** Árbol de Decisión:** Diferencia = 0.0563 → Overfitting
** Regresión Logística:** Diferencia = 0.0048 → Excelente (NO overfitting)
** SVM (Lineal):** Diferencia = 0.0070 → Excelente (NO overfitting)


---

**Referencia:**

- Diferencia < 0.02 → Excelente (NO overfitting)

- Diferencia 0.02-0.05 → Aceptable

- Diferencia > 0.05 → Overfitting


 Modelo con menor sobreajuste:  Regresión Logística


## Guardado de resultados para importar en optimización

In [51]:
ruta_pkl = "/content/drive/MyDrive/Ciencia de Datos Collab/Programación para la ciencia de datos/Evaluacion_2/Processed/resultados_modelos.pkl"

variables_guardar = {
    'accuracy_dt': accuracy_dt,
    'accuracy_lr': accuracy_lr,
    'accuracy_svm': accuracy_svm,
    'f1_dt': f1_dt,
    'f1_lr': f1_lr,
    'f1_svm': f1_svm,
    'y_pred_dt': y_pred_dt,
    'y_pred_lr': y_pred_lr,
    'y_pred_svm': y_pred_svm,
    'y_test': y_test,
    'le': le
}

with open(ruta_pkl, 'wb') as f:
    pickle.dump(variables_guardar, f)

print("Variables guardadas en Drive")

Variables guardadas en Drive
